In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
office_table = dbutils.widgets.get("office_table")
client_table = dbutils.widgets.get("client_table")
date_table = dbutils.widgets.get("date_table")
paymentdetail_table = dbutils.widgets.get("paymentdetail_table")
paymenttype_table = dbutils.widgets.get("paymenttype_table")

In [0]:
display(
    spark.sql(
        f"""
        CREATE OR REPLACE TEMP VIEW transactions_src AS
        SELECT
            CAST(ReportingDate AS DATE) AS ReportingDate,
            CAST(PaymentID AS BIGINT) AS PaymentID,
            CAST(FacilityCode AS INT) AS FacilityCode,
            CAST(AcctNbr AS STRING) AS AcctNbr,
            CAST(TransDate AS INT) AS TransDate,
            CAST(EntryDate AS INT) AS EntryDate,
            CAST(TransAmt AS DOUBLE) AS TransAmt,
            CAST(TransCode AS STRING) AS TransCode,
            CAST(TransDesc AS STRING) AS TransDesc,
            CAST(TransType AS STRING) AS TransType,
            NULL AS InsCode,
            NULL AS Payor,
            NULL AS FinClass,
            NULL AS Coinsurance,
            NULL AS Deductible,
            NULL AS CoPay,
            NULL AS PatResp,
            CAST(BatchID AS STRING) AS BatchID,
            CAST(SourceSystemKey AS INT) AS SourceSystemKey,
            current_timestamp() AS _load_timestamp
        FROM (
            WITH 
            transactions_cte AS (
                SELECT
                    to_date(CAST(date_entered_key AS STRING), 'yyyyMMdd') AS ReportingDate,
                    (
                        CAST(date_entered_key AS BIGINT) * 1000000
                        + ROW_NUMBER() OVER (ORDER BY bcf.posted_date_key)
                    ) AS PaymentID,
                    ofc.OfficeNumber AS FacilityCode,
                    CASE
                        WHEN UPPER(bcf.invoice_number) = 'ADV'
                            THEN CONCAT('ADV', ' - ', clt.SourceSystemId)
                        ELSE bcf.invoice_number
                    END AS AcctNbr,
                    bcf.deposit_date_key AS TransDate,
                    bcf.posted_date_key AS EntryDate,
                    bcf.cash_collected      AS TransAmt,
                    CASE
                        WHEN pt.PaymentTypeDescription LIKE '%|%'
                            THEN REPLACE(pt.PaymentTypeDescription, '|', '')
                        ELSE pt.PaymentTypeDescription
                    END AS TransCode,
                    CASE
                        WHEN bcf.payment_number LIKE '%|%'
                            THEN REPLACE(bcf.payment_number, '|', '')
                        ELSE bcf.payment_number
                    END AS TransDesc,
                    pt.TransactionType AS TransType,
                    pd.BatchID,
                    0 AS SourceSystemKey,
                    ROW_NUMBER() OVER (
                      PARTITION BY
                        CASE
                            WHEN UPPER(bcf.invoice_number) = 'ADV'
                                THEN CONCAT('ADV', ' - ', clt.SourceSystemId)
                            ELSE bcf.invoice_number
                        END
                      ORDER BY
                        CASE
                            WHEN UPPER(bcf.invoice_number) = 'ADV'
                                THEN CONCAT('ADV', ' - ', clt.SourceSystemId)
                            ELSE bcf.invoice_number
                        END
                    ) AS rn
                FROM {source_table} bcf
                LEFT JOIN {office_table} ofc
                    ON bcf.office_key = ofc.OfficeKey
                LEFT JOIN {paymenttype_table} pt
                    ON pt.PaymentTypeKey = bcf.payment_type_key
                LEFT JOIN {paymentdetail_table} pd
                    ON pd.PaymentDetailKey = bcf.payment_detail_key
                JOIN {date_table} dt
                    ON dt.DateKey = bcf.posted_date_key
                LEFT JOIN {client_table} clt
                    ON clt.ClientKey = bcf.client_key
                WHERE bcf.date_entered_key = date_format(DATE('{fetch_date}'), 'yyyyMMdd')
            )
            SELECT 
                ReportingDate,
                PaymentID,
                FacilityCode,
                AcctNbr,
                TransDate,
                EntryDate,
                TransAmt,
                TransCode,
                TransDesc,
                TransType,
                BatchID,
                SourceSystemKey
            FROM transactions_cte
            WHERE rn=1
        ) AS src
        """
    )
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING transactions_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 0

WHEN MATCHED THEN
UPDATE SET
    tgt.PaymentID = src.PaymentID,
    tgt.FacilityCode = src.FacilityCode,
    tgt.TransDate = src.TransDate,
    tgt.EntryDate = src.EntryDate,
    tgt.TransAmt = src.TransAmt,
    tgt.TransCode = src.TransCode,
    tgt.TransDesc = src.TransDesc,
    tgt.TransType = src.TransType,
    tgt.InsCode = src.InsCode,
    tgt.Payor = src.Payor,
    tgt.FinClass = src.FinClass,
    tgt.Coinsurance = src.Coinsurance,
    tgt.Deductible = src.Deductible,
    tgt.CoPay = src.CoPay,
    tgt.PatResp = src.PatResp,
    tgt.BatchID = src.BatchID,
    tgt.SourceSystemKey = src.SourceSystemKey,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    PaymentID,
    FacilityCode,
    AcctNbr,
    TransDate,
    EntryDate,
    TransAmt,
    TransCode,
    TransDesc,
    TransType,
    InsCode,
    Payor,
    FinClass,
    Coinsurance,
    Deductible,
    CoPay,
    PatResp,
    BatchID,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.PaymentID,
    src.FacilityCode,
    src.AcctNbr,
    src.TransDate,
    src.EntryDate,
    src.TransAmt,
    src.TransCode,
    src.TransDesc,
    src.TransType,
    src.InsCode,
    src.Payor,
    src.FinClass,
    src.Coinsurance,
    src.Deductible,
    src.CoPay,
    src.PatResp,
    src.BatchID,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)